In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)

from xgboost import XGBRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error
from prophet import Prophet

In [11]:
df = pd.read_csv("retail_sales.csv")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")
df = df.drop_duplicates("date")

df = df.set_index("date").asfreq("D")

# interpolate missing sales
df["sales"] = df["sales"].interpolate(method="time")
df["sales"] = df["sales"].ffill().bfill()

sales = df["sales"]

print("Dataset Shape :", df.shape)
print("Start Date    :", df.index.min())
print("End Date      :", df.index.max())
print("Missing Date  :", sales.isna().sum())

Dataset Shape : (1461, 4)
Start Date    : 2021-01-01 00:00:00
End Date      : 2024-12-31 00:00:00
Missing Date  : 0


In [12]:
# Hold out the final 56 days
holdout = 56

train = df.iloc[:-holdout].copy()
test = df.iloc[-holdout:].copy()

test_sales = test["sales"]

print("Training Data Shape :", train.shape)
print("Holdout Data Shape  :", test.shape)
print("Training End Date   :", train.index.max())
print("Holdout Start Date  :", test.index.min())
print("Holdout End Date    :", test.index.max())

Training Data Shape : (1405, 4)
Holdout Data Shape  : (56, 4)
Training End Date   : 2024-11-05 00:00:00
Holdout Start Date  : 2024-11-06 00:00:00
Holdout End Date    : 2024-12-31 00:00:00
